# AFS-VFM Phase 2B - Internal Signal Probing

This notebook extracts **internal model representations** from DINOv2, CLIP, and DETR
during inference on degraded images. Unlike Phase 1 (which only captured the final
prediction), this captures embeddings, attention weights, logit distributions, and
confidence scores at every degradation frame.

**GPU Required:** Use Kaggle T4 GPU or Colab T4 GPU.

**Before running:** Set runtime to GPU:
- Kaggle: `Settings > Accelerator > GPU T4 x2`
- Colab: `Runtime > Change runtime type > T4 GPU`

**Estimated time:** ~2-3 hours on T4 GPU (300 images x 5 degradations x 20 frames x 3 models = 90,000 inferences)

In [ ]:
# ── Step 1: Check GPU ──
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Set Accelerator to GPU T4.")

In [ ]:
# ── Step 2: Clone repo and install dependencies ──
!git clone https://github.com/AmanDevNet/AFS-VFM.git
%cd AFS-VFM
!pip install -q -r requirements.txt timm datasets huggingface_hub scipy

In [ ]:
# ── Step 3: HuggingFace login (for ImageNet access) ──
from huggingface_hub import login
login(token="PASTE_YOUR_HF_TOKEN_HERE")  # <-- Replace with your actual token

In [ ]:
# ── Step 4: Download datasets ──
!python -u download_dataset.py

In [ ]:
# ── Step 5: Upload Phase 2A results ──
# Upload your selected_images.csv and image_list.txt from results/phase2b/
# If running on Kaggle, you can add them as a dataset.
# For Colab, upload manually:
import os
os.makedirs('results/phase2b', exist_ok=True)

# Option A: If files are uploaded to the notebook
# from google.colab import files
# uploaded = files.upload()  # Upload selected_images.csv

# Option B: If you want to skip subset selection and probe ALL images
# (takes much longer ~8-10 hours), set USE_FULL_SET = True
USE_FULL_SET = False
SUBSET_SIZE = 300

print(f"Mode: {'FULL SET (all images)' if USE_FULL_SET else f'SUBSET ({SUBSET_SIZE} images)'}")

In [ ]:
# ── Step 6: Select images to probe ──
import sys
import random
import pandas as pd
sys.path.insert(0, 'src')

COCO_DIR = 'data/coco_val'
IMAGENET_DIR = 'data/imagenet_val'

# Collect all available images
all_images = []
for directory, label in [(COCO_DIR, 'coco'), (IMAGENET_DIR, 'imagenet')]:
    if os.path.isdir(directory):
        for f in sorted(os.listdir(directory)):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_images.append((os.path.join(directory, f), label))
print(f"Total available images: {len(all_images)}")

if USE_FULL_SET:
    probe_images = all_images
elif os.path.exists('results/phase2b/image_list.txt'):
    # Use the strategically selected subset from Phase 2A
    with open('results/phase2b/image_list.txt') as f:
        selected = set(line.strip() for line in f if line.strip())
    probe_images = [(p, ds) for p, ds in all_images if os.path.basename(p) in selected]
    print(f"Using Phase 2A strategic subset: {len(probe_images)} images")
else:
    # Fallback: random sample
    random.seed(42)
    probe_images = random.sample(all_images, min(SUBSET_SIZE, len(all_images)))
    print(f"Using random subset: {len(probe_images)} images")

print(f"\nImages to probe: {len(probe_images)}")
print(f"Total inferences: {len(probe_images) * 5 * 20 * 3:,}")

In [ ]:
# ── Step 7: Load all 3 models ──
from models.model_loader import VisionModelLoader
from models.signal_extractor import SignalExtractor

loader = VisionModelLoader()
device = loader.device

dino_model, dino_proc = loader.load_dinov2()
clip_model, clip_proc = loader.load_clip()
detr_model, detr_proc = loader.load_detr()

extractor = SignalExtractor(device)
print("\nAll models loaded. Signal extractor ready.")

In [ ]:
# ── Step 8: Run Signal Probing (main loop) ──
import csv
import time
import numpy as np
from PIL import Image
from degradation import generate_degradation_sequence

DEGRADATION_TYPES = ['blur', 'occlusion', 'lighting', 'scale', 'viewpoint']
NUM_FRAMES = 20
CLIP_LABELS = [
    'a dog', 'a cat', 'a car', 'a bicycle', 'an airplane',
    'a clock', 'fruit', 'a person', 'a coffee cup', 'a traffic scene',
    'a bird', 'a boat', 'a horse', 'a chair', 'a bottle',
    'a bus', 'a truck', 'a train', 'a flower', 'a tree',
]

os.makedirs('results/phase2b/embeddings', exist_ok=True)
os.makedirs('results/phase2b/signals', exist_ok=True)

# CSV output for frame-level computed metrics
signals_csv_path = 'results/phase2b/signals/frame_signals.csv'
signal_fields = [
    'image', 'dataset', 'degradation', 'frame', 'severity_pct',
    # DINOv2 signals
    'dinov2_prediction', 'dinov2_top1_confidence', 'dinov2_logit_margin',
    'dinov2_attention_entropy', 'dinov2_cosine_sim', 'dinov2_l2_distance',
    'dinov2_top5_overlap',
    # CLIP signals
    'clip_prediction', 'clip_top1_confidence', 'clip_logit_margin',
    'clip_cosine_sim', 'clip_l2_distance',
    # DETR signals
    'detr_prediction', 'detr_top_detection_confidence', 'detr_objectness',
    'detr_num_detections',
]

# Resume support
completed_images = set()
all_rows = []
if os.path.exists(signals_csv_path):
    existing = pd.read_csv(signals_csv_path)
    from collections import Counter
    img_counts = Counter(existing['image'])
    expected = len(DEGRADATION_TYPES) * NUM_FRAMES
    for img_name, count in img_counts.items():
        if count >= expected:
            completed_images.add(img_name)
    all_rows = existing.to_dict('records')
    print(f"Resuming: {len(completed_images)} images already done")

remaining = [(p, ds) for p, ds in probe_images if os.path.basename(p) not in completed_images]
total_images = len(probe_images)

# Storage for embeddings (saved per image to manage memory)
print(f"\nStarting signal probing...")
print(f"  Total: {total_images}, Already done: {len(completed_images)}, Remaining: {len(remaining)}")
print(f"  Inferences remaining: {len(remaining) * 5 * 20 * 3:,}")
print()

start_time = time.time()
embeddings_store = {}  # {image: {model: {frame: embedding}}}

for img_idx, (img_path, dataset_source) in enumerate(remaining, len(completed_images) + 1):
    img_file = os.path.basename(img_path)
    print(f"\n{'='*60}")
    print(f"  Image [{img_idx}/{total_images}]: {img_file} ({dataset_source})")
    print(f"{'='*60}")

    img_embeddings = {'dinov2': {}, 'clip': {}}

    for deg_type in DEGRADATION_TYPES:
        print(f"  >> {deg_type}", end=' ')

        try:
            frames = generate_degradation_sequence(img_path, deg_type, num_frames=NUM_FRAMES)
        except Exception as exc:
            print(f"ERROR: {exc}")
            continue

        # Baselines (frame 0 signals)
        baseline_dinov2 = None
        baseline_clip = None

        for frame_idx in range(NUM_FRAMES):
            frame = frames[frame_idx]
            severity_pct = int(frame_idx / (NUM_FRAMES - 1) * 100)

            if isinstance(frame, np.ndarray):
                frame_pil = Image.fromarray(frame)
            else:
                frame_pil = frame

            row = {
                'image': img_file,
                'dataset': dataset_source,
                'degradation': deg_type,
                'frame': frame_idx + 1,
                'severity_pct': severity_pct,
            }

            # ---- DINOv2 ----
            try:
                dino_sig = extractor.extract_dinov2(dino_model, dino_proc, frame_pil)
                row['dinov2_prediction'] = dino_sig['prediction']
                row['dinov2_top1_confidence'] = dino_sig['top1_confidence']
                row['dinov2_logit_margin'] = dino_sig['logit_margin']
                row['dinov2_attention_entropy'] = dino_sig['attention_entropy']

                if frame_idx == 0:
                    baseline_dinov2 = dino_sig
                    row['dinov2_cosine_sim'] = 1.0
                    row['dinov2_l2_distance'] = 0.0
                    row['dinov2_top5_overlap'] = 1.0
                else:
                    row['dinov2_cosine_sim'] = extractor.cosine_similarity(
                        baseline_dinov2['cls_embedding'], dino_sig['cls_embedding'])
                    row['dinov2_l2_distance'] = extractor.l2_distance(
                        baseline_dinov2['cls_embedding'], dino_sig['cls_embedding'])
                    row['dinov2_top5_overlap'] = extractor.top5_overlap(
                        baseline_dinov2['top5_labels'], dino_sig['top5_labels'])

                # Store embedding for this degradation's first frame only
                if deg_type == 'blur':  # store one degradation's embeddings to save memory
                    img_embeddings['dinov2'][frame_idx] = dino_sig['cls_embedding']
            except Exception as e:
                row['dinov2_prediction'] = 'ERROR'
                for k in ['dinov2_top1_confidence', 'dinov2_logit_margin',
                          'dinov2_attention_entropy', 'dinov2_cosine_sim',
                          'dinov2_l2_distance', 'dinov2_top5_overlap']:
                    row[k] = float('nan')

            # ---- CLIP ----
            try:
                clip_sig = extractor.extract_clip(clip_model, clip_proc, frame_pil, CLIP_LABELS)
                row['clip_prediction'] = clip_sig['prediction']
                row['clip_top1_confidence'] = clip_sig['top1_confidence']
                row['clip_logit_margin'] = clip_sig['logit_margin']

                if frame_idx == 0:
                    baseline_clip = clip_sig
                    row['clip_cosine_sim'] = 1.0
                    row['clip_l2_distance'] = 0.0
                else:
                    row['clip_cosine_sim'] = extractor.cosine_similarity(
                        baseline_clip['image_embedding'], clip_sig['image_embedding'])
                    row['clip_l2_distance'] = extractor.l2_distance(
                        baseline_clip['image_embedding'], clip_sig['image_embedding'])

                if deg_type == 'blur':
                    img_embeddings['clip'][frame_idx] = clip_sig['image_embedding']
            except Exception as e:
                row['clip_prediction'] = 'ERROR'
                for k in ['clip_top1_confidence', 'clip_logit_margin',
                          'clip_cosine_sim', 'clip_l2_distance']:
                    row[k] = float('nan')

            # ---- DETR ----
            try:
                detr_sig = extractor.extract_detr(detr_model, detr_proc, frame_pil)
                row['detr_prediction'] = detr_sig['prediction']
                row['detr_top_detection_confidence'] = detr_sig['top_detection_confidence']
                row['detr_objectness'] = detr_sig['objectness_score']
                row['detr_num_detections'] = detr_sig['num_detections_above_threshold']
            except Exception as e:
                row['detr_prediction'] = 'ERROR'
                for k in ['detr_top_detection_confidence', 'detr_objectness',
                          'detr_num_detections']:
                    row[k] = float('nan')

            all_rows.append(row)

        print(f"[done]")

    # Save embeddings for this image (blur degradation only, to manage disk)
    emb_path = f"results/phase2b/embeddings/{img_file.replace('.', '_')}_embeddings.npz"
    np.savez_compressed(
        emb_path,
        **{f'dinov2_frame{k}': v for k, v in img_embeddings['dinov2'].items()},
        **{f'clip_frame{k}': v for k, v in img_embeddings['clip'].items()},
    )

    # Save CSV after every image (crash-safe)
    df_out = pd.DataFrame(all_rows)
    df_out.to_csv(signals_csv_path, index=False)

    elapsed = time.time() - start_time
    imgs_done = img_idx - len(completed_images)
    rate = elapsed / imgs_done if imgs_done else 0
    remaining_est = rate * (len(remaining) - imgs_done)
    print(f"  Saved! [{img_idx}/{total_images}] ~{remaining_est/60:.0f} min remaining")

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"  SIGNAL PROBING COMPLETE")
print(f"  Images processed: {total_images}")
print(f"  Total signal rows: {len(all_rows):,}")
print(f"  Time: {elapsed/3600:.1f} hours")
print(f"{'='*60}")

In [ ]:
# ── Step 9: Quick summary of extracted signals ──
df = pd.read_csv(signals_csv_path)
print(f"Total rows: {len(df):,}")
print(f"Unique images: {df['image'].nunique()}")
print(f"\nDINOv2 confidence: mean={df['dinov2_top1_confidence'].mean():.4f}, "
      f"min={df['dinov2_top1_confidence'].min():.4f}")
print(f"DINOv2 cosine sim: mean={df['dinov2_cosine_sim'].mean():.4f}, "
      f"min={df['dinov2_cosine_sim'].min():.4f}")
print(f"DINOv2 attention entropy: mean={df['dinov2_attention_entropy'].mean():.4f}")
print(f"\nCLIP confidence: mean={df['clip_top1_confidence'].mean():.4f}")
print(f"CLIP cosine sim: mean={df['clip_cosine_sim'].mean():.4f}")
print(f"\nDETR detection conf: mean={df['detr_top_detection_confidence'].mean():.4f}")
print(f"DETR objectness: mean={df['detr_objectness'].mean():.4f}")
print(f"DETR detections above 0.5: mean={df['detr_num_detections'].mean():.1f}")

In [ ]:
# ── Step 10: Download results ──
# For Colab:
# from google.colab import files
# files.download('results/phase2b/signals/frame_signals.csv')

# For Kaggle: results will be in the output directory
import shutil
shutil.make_archive('phase2b_results', 'zip', 'results/phase2b')
print("Results packed into phase2b_results.zip")
print(f"Signal CSV size: {os.path.getsize(signals_csv_path) / 1e6:.1f} MB")
print(f"Embedding files: {len(os.listdir('results/phase2b/embeddings'))}")
print("\nDownload phase2b_results.zip from the Output tab.")